In [1]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    from google.colab import userdata, files

    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
        print('Upload your kaggle.json')
        uploaded = files.upload()
        shutil.move('kaggle.json', f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)

    !pip install -q kaggle mlflow wandb

    import wandb
    wandb.login(key=userdata.get('WANDB_API_KEY'))

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
    !unzip -o -q data/walmart-recruiting-store-sales-forecasting.zip -d data
    !for f in data/*.zip; do unzip -o -q "$f" -d data; done

    !mlflow db upgrade sqlite:///mlflow.db

    DATA_DIR = 'data'
else:
    DATA_DIR = '..'

Upload your kaggle.json


Saving kaggle.json to kaggle.json
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adola23 (adola23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100% 2.70M/2.70M [00:00<00:00, 122MB/s]

2026/07/11 12:20:37 INFO mlflow.store.db.utils: Updating database tables


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.base import BaseEstimator
from sklearn.pipeline import Pipeline
import mlflow
import mlflow.sklearn
import wandb

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cpu


In [3]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['Date'])
train.shape

(421570, 5)

In [4]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)

In [5]:
class WindowDataset(Dataset):
    def __init__(self, windows, targets, weights):
        self.windows = windows
        self.targets = targets
        self.weights = weights

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return self.windows[idx], self.targets[idx], self.weights[idx]

def build_series_matrix(df):
    pivot = df.pivot_table(index=['Store', 'Dept'], columns='Date', values='Weekly_Sales')
    pivot = pivot.sort_index(axis=1)
    holiday = df.drop_duplicates('Date').set_index('Date')['IsHoliday'].sort_index()
    holiday = holiday.reindex(pivot.columns).fillna(False)
    return pivot, holiday

def make_windows(pivot, holiday, seq_len=52):
    values = pivot.fillna(0).values
    n_series, n_time = values.shape

    windows, targets, weights = [], [], []
    for t in range(seq_len, n_time):
        window = values[:, t - seq_len:t]
        target = values[:, t]
        valid = ~pivot.iloc[:, t].isna().values
        if valid.sum() == 0:
            continue
        w = np.where(holiday.iloc[t], 5.0, 1.0)
        windows.append(window[valid])
        targets.append(target[valid])
        weights.append(np.full(valid.sum(), w))
    return np.concatenate(windows), np.concatenate(targets), np.concatenate(weights)

In [6]:
# N-BEATS (Oreshkin et al. 2019): fully-connected blockebis stack. titoeuli blocki
# input window-dan itvlis backcast-s (ras xsnis is warsulshi) da forecast-s.
# shemdeg blocks gadaecema mxolod residuali (window minus backcast), prognozebi
# ki ikribeba. DLinear-is sapirispiro fsonia: 670 atasamde parametri 106-is nacvlad
class NBeatsBlock(nn.Module):
    def __init__(self, input_size, pred_len, hidden, n_layers):
        super().__init__()
        layers = []
        for i in range(n_layers):
            layers.append(nn.Linear(input_size if i == 0 else hidden, hidden))
            layers.append(nn.ReLU())
        self.fc = nn.Sequential(*layers)
        self.backcast = nn.Linear(hidden, input_size)
        self.forecast = nn.Linear(hidden, pred_len)

    def forward(self, x):
        h = self.fc(x)
        return self.backcast(h), self.forecast(h)

class NBeats(nn.Module):
    def __init__(self, seq_len, pred_len=1, n_blocks=3, hidden=256, n_layers=4):
        super().__init__()
        self.blocks = nn.ModuleList(
            [NBeatsBlock(seq_len, pred_len, hidden, n_layers) for _ in range(n_blocks)])

    def forward(self, x):
        residual = x
        forecast = torch.zeros(x.shape[0], 1, device=x.device)
        for block in self.blocks:
            back, fore = block(residual)
            residual = residual - back
            forecast = forecast + fore
        return forecast

class NBeatsForecaster(BaseEstimator):
    def __init__(self, seq_len=52, epochs=10, lr=1e-3, batch_size=1024, n_blocks=3, hidden=256, n_layers=4):
        self.seq_len = seq_len
        self.epochs = epochs
        self.lr = lr
        self.batch_size = batch_size
        self.n_blocks = n_blocks
        self.hidden = hidden
        self.n_layers = n_layers

    def _build_net(self):
        return NBeats(self.seq_len_, 1, self.n_blocks, self.hidden, self.n_layers)

    def fit(self, X, y, log_fn=None):
        df = X.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df['Weekly_Sales'] = y.values if hasattr(y, 'values') else y

        pivot, holiday = build_series_matrix(df)
        self.seq_len_ = min(self.seq_len, pivot.shape[1] - 1)
        self.series_mean_ = pivot.mean(axis=1).fillna(0)
        self.series_std_ = pivot.std(axis=1).fillna(1).replace(0, 1)

        norm_pivot = pivot.sub(self.series_mean_, axis=0).div(self.series_std_, axis=0)
        windows, targets, weights = make_windows(norm_pivot, holiday, self.seq_len_)

        X_t = torch.tensor(windows, dtype=torch.float32)
        y_t = torch.tensor(targets, dtype=torch.float32).unsqueeze(1)
        w_t = torch.tensor(weights, dtype=torch.float32).unsqueeze(1)

        loader = DataLoader(WindowDataset(X_t, y_t, w_t), batch_size=self.batch_size, shuffle=True)

        self.model_ = self._build_net().to(DEVICE)
        opt = torch.optim.Adam(self.model_.parameters(), lr=self.lr)

        for epoch in range(self.epochs):
            epoch_loss = 0.0
            for xb, yb, wb in loader:
                xb, yb, wb = xb.to(DEVICE), yb.to(DEVICE), wb.to(DEVICE)
                opt.zero_grad()
                pred = self.model_(xb)
                loss = (wb * (pred - yb).abs()).mean()
                loss.backward()
                opt.step()
                epoch_loss += loss.item() * len(xb)
            epoch_loss /= len(loader.dataset)
            if log_fn is not None:
                log_fn(epoch, epoch_loss)

        self.model_ = self.model_.cpu()
        self.history_ = pivot
        return self

    def _forecast_series(self, key, target_dates):
        if key not in self.history_.index:
            return pd.Series(self.series_mean_.mean(), index=target_dates)

        mean = self.series_mean_.loc[key]
        std = self.series_std_.loc[key]
        series = self.history_.loc[key].copy()

        max_date = max(target_dates.max(), series.index.max())
        all_dates = pd.date_range(series.index.min(), max_date, freq='W-FRI')

        series = series.reindex(all_dates)
        known_mask = series.notna()
        series = series.fillna(0)

        values = ((series - mean) / std).values.tolist()

        self.model_.eval()
        with torch.no_grad():
            for i in range(len(values)):
                if i >= self.seq_len_ and not known_mask.iloc[i]:
                    window = torch.tensor([values[i - self.seq_len_:i]], dtype=torch.float32)
                    values[i] = self.model_(window).item()

        result = pd.Series(values, index=all_dates) * std + mean
        return result.reindex(target_dates)

    def predict(self, X):
        df = X.copy().reset_index(drop=True)
        df['Date'] = pd.to_datetime(df['Date'])
        preds = np.zeros(len(df))
        for (store, dept), group in df.groupby(['Store', 'Dept']):
            forecast = self._forecast_series((store, dept), pd.DatetimeIndex(group['Date'].unique()))
            preds[group.index] = forecast.reindex(group['Date']).values
        return preds

In [7]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('NBEATS_Training')

2026/07/11 12:20:55 INFO mlflow.tracking.fluent: Experiment with name 'NBEATS_Training' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/walmart-sales-forecasting/mlruns/12', creation_time=1783772455509, effective_trace_archival_retention=None, experiment_id='12', last_update_time=1783772455509, lifecycle_stage='active', name='NBEATS_Training', tags={}, trace_location=None, workspace='default'>

In [8]:
with mlflow.start_run(run_name='NBEATS_Cleaning'):
    wandb.init(project='walmart-sales-forecasting', name='NBEATS_Cleaning', reinit='finish_previous')

    n_negative = int((train['Weekly_Sales'] < 0).sum())
    mlflow.log_param('negative_sales_rows', n_negative)
    mlflow.log_param('negative_sales_action', 'clip_to_zero')
    wandb.log({'negative_sales_rows': n_negative})
    wandb.finish()

    y_train = train['Weekly_Sales'].clip(lower=0)

negative_sales_rows,▁
negative_sales_rows,1285


In [9]:
with mlflow.start_run(run_name='NBEATS_Windowing'):
    mlflow.log_param('seq_len', 52)
    pivot, holiday = build_series_matrix(train.assign(Weekly_Sales=y_train))
    windows, targets, weights = make_windows(pivot, holiday, 52)
    mlflow.log_metric('n_windows', len(windows))
    mlflow.log_metric('n_series', len(pivot))

len(windows), pivot.shape

(269196, (3331, 143))

In [10]:
def time_based_splits(dates, n_splits=3):
    dates = np.sort(dates.unique())
    fold_size = len(dates) // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_end = dates[fold_size * i - 1]
        val_end = dates[min(fold_size * (i + 1) - 1, len(dates) - 1)]
        splits.append((train_end, val_end))
    return splits

splits = time_based_splits(train['Date'])
splits

# test-formis holdout: oqtombramde vswavlobt, shemdegi noembridan ivlisamde vafasebt.
# diagnostikam achvena rom es fanjara kaggle-is tests bevrad ukot imeorebs vidre CV
HOLDOUT_CUTOFF = pd.Timestamp('2011-10-28')
HOLDOUT_END = pd.Timestamp('2012-07-27')

In [11]:
params = dict(seq_len=52, n_blocks=3, hidden=256, n_layers=4, epochs=10, lr=1e-3, batch_size=1024)

with mlflow.start_run(run_name='NBEATS_CV'):
    mlflow.log_params(params)
    wandb.init(project='walmart-sales-forecasting', name='NBEATS_CV', config=params, reinit='finish_previous')

    fold_scores = []
    for i, (train_end, val_end) in enumerate(splits):
        tm = train['Date'] <= train_end
        vm = (train['Date'] > train_end) & (train['Date'] <= val_end)

        def log_fn(epoch, loss, fold=i):
            mlflow.log_metric(f'train_loss_fold{fold}', loss, step=epoch)
            wandb.log({f'train_loss_fold{fold}': loss})

        model = NBeatsForecaster(**params)
        model.fit(train.loc[tm, ['Store', 'Dept', 'Date', 'IsHoliday']], y_train[tm], log_fn=log_fn)
        preds = model.predict(train.loc[vm, ['Store', 'Dept', 'Date', 'IsHoliday']])

        score = wmae(y_train[vm].values, preds, train.loc[vm, 'IsHoliday'].values)
        fold_scores.append(score)
        mlflow.log_metric('wmae', score, step=i)
        wandb.log({'fold': i, 'wmae': score})

    hm = train['Date'] <= HOLDOUT_CUTOFF
    hv = (train['Date'] > HOLDOUT_CUTOFF) & (train['Date'] <= HOLDOUT_END)
    hmodel = NBeatsForecaster(**params)
    hmodel.fit(train.loc[hm, ['Store', 'Dept', 'Date', 'IsHoliday']], y_train[hm])
    hpreds = hmodel.predict(train.loc[hv, ['Store', 'Dept', 'Date', 'IsHoliday']])
    wmae_holdout = wmae(y_train[hv].values, hpreds, train.loc[hv, 'IsHoliday'].values)

    mlflow.log_metric('wmae_mean', float(np.mean(fold_scores)))
    mlflow.log_metric('wmae_std', float(np.std(fold_scores)))
    mlflow.log_metric('wmae_holdout', wmae_holdout)
    wandb.log({'wmae_mean': float(np.mean(fold_scores)), 'wmae_std': float(np.std(fold_scores)),
               'wmae_holdout': wmae_holdout})
    wandb.finish()

fold_scores, wmae_holdout

fold,▁▅█
train_loss_fold0,█▆▅▄▃▂▂▂▁▁
train_loss_fold1,█▆▅▄▄▃▃▂▂▁
train_loss_fold2,█▆▅▄▃▃▂▂▁▁
wmae,█▇▁
wmae_holdout,▁
wmae_mean,▁
wmae_std,▁
fold,2
train_loss_fold0,0.41681
train_loss_fold1,0.38033


([np.float64(4244.632173700676),
  np.float64(3871.579671298777),
  np.float64(2184.4498626043796)],
 np.float64(3482.275318534498))

In [12]:
with mlflow.start_run(run_name='NBEATS_Final'):
    mlflow.log_params(params)
    wandb.init(project='walmart-sales-forecasting', name='NBEATS_Final', config=params, reinit='finish_previous')

    def log_fn(epoch, loss):
        mlflow.log_metric('train_loss', loss, step=epoch)
        wandb.log({'epoch': epoch, 'train_loss': loss})

    pipeline = Pipeline([
        ('model', NBeatsForecaster(**params)),
    ])
    pipeline.fit(train[['Store', 'Dept', 'Date', 'IsHoliday']], y_train, model__log_fn=log_fn)

    mlflow.sklearn.log_model(pipeline, name='model', serialization_format='cloudpickle')
    wandb.finish()

2026/07/11 12:39:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/11 12:39:27 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▆▅▄▄▃▃▂▁▁
epoch,9
train_loss,0.45473


In [13]:
if IN_COLAB:
    import requests
    gh_user = requests.get('https://api.github.com/user', headers={'Authorization': f'token {github_token}'}).json()['login']
    !git config user.name "{gh_user}"
    !git config user.email "{gh_user}@users.noreply.github.com"

    !git add mlflow.db mlruns
    !git commit -m "Run model_experiment_NBEATS.ipynb in Colab"
    !git push

[main 259b8d5] Run model_experiment_NBEATS.ipynb in Colab
 6 files changed, 61 insertions(+)
 create mode 100644 mlruns/12/models/m-70dd2c45d8b540e99efcabcbff6b8774/artifacts/MLmodel
 create mode 100644 mlruns/12/models/m-70dd2c45d8b540e99efcabcbff6b8774/artifacts/conda.yaml
 create mode 100644 mlruns/12/models/m-70dd2c45d8b540e99efcabcbff6b8774/artifacts/model.pkl
 create mode 100644 mlruns/12/models/m-70dd2c45d8b540e99efcabcbff6b8774/artifacts/python_env.yaml
 create mode 100644 mlruns/12/models/m-70dd2c45d8b540e99efcabcbff6b8774/artifacts/requirements.txt
Enumerating objects: 16, done.
Counting objects: 100% (16/16), done.
Delta compression using up to 2 threads
Compressing objects: 100% (11/11), done.
Writing objects: 100% (13/13), 4.17 MiB | 4.17 MiB/s, done.
Total 13 (delta 3), reused 3 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/giomamaca/walmart-sales-forecasting.git
   784753d..259b8d5  main -> main
